In [4]:
# Import the Pre-requisites libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

In [2]:
# Load the CSV file
df = pd.read_csv(r"C:\DTT ML 2026\03-classification\course_lead_scoring_2026.csv")

In [3]:
# Check that the dataset loaded properly and view the first few rows
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [9]:
# Step 2: Data Preparation
# Check missing values, then impute categorical columns with 'NA' and numerical columns with 0.0

# Identify numerical and categorical columns
categorical = list(df.select_dtypes(include=['object', 'string']).columns)
numerical = list(df.select_dtypes(include=['number']).columns)

# If target 'converted' is in numerical, make sure it remains untouched
if 'converted' in numerical:
    numerical.remove('converted')

print("Categorical features:", categorical)
print("Numerical features:", numerical)

# Imputation according to instructions
for col in categorical:
    df[col] = df[col].fillna('NA')

for col in numerical:
    df[col] = df[col].fillna(0.0)

print("\nMissing values remaining:")
print(df.isnull().sum())

Categorical features: ['lead_source', 'industry', 'employment_status', 'location']
Numerical features: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

Missing values remaining:
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [11]:
# Question 1: Mode for industry
# Find the most frequent value (mode) in the industry column

mode_industry = df['industry'].mode()[0]
val_counts = df['industry'].value_counts()

print("Mode of industry:", mode_industry)
print("\nTop value counts for industry:")
print(val_counts)

Mode of industry: technology

Top value counts for industry:
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


In [12]:
# Question 2: Correlation Matrix for Numerical Features
# Compute Pearson correlation across numerical features and evaluate the four specified pairs

corr_matrix = df[numerical].corr().abs()

candidate_pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

print("Correlation pairs comparison:")
for f1, f2 in candidate_pairs:
    val = corr_matrix.loc[f1, f2]
    print(f"  {f1} & {f2}: {val:.4f}")


Correlation pairs comparison:
  interaction_count & lead_score: 0.9157
  number_of_courses_viewed & lead_score: 0.7572
  number_of_courses_viewed & interaction_count: 0.7216
  annual_income & interaction_count: 0.1228


In [13]:
# Data Split : Train / Val / Test Split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [14]:
# Question 3: Mutual Information on training set

def calculate_mi(series):
    return mutual_info_score(series, y_train)

mi_scores = df_train[categorical].apply(calculate_mi)
mi_scores = mi_scores.round(2).sort_values(ascending=False)

print("Mutual Information Scores (rounded to 2 decimals):")
print(mi_scores)

Mutual Information Scores (rounded to 2 decimals):
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


In [16]:
# Question 4: Baseline Logistic Regression

train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
original_acc = accuracy_score(y_val, y_pred)

print(f"Validation Accuracy: {original_acc:.4f}")
print(f"Rounded (2 decimals): {round(original_acc, 2)}")

Validation Accuracy: 0.6450
Rounded (2 decimals): 0.65


In [22]:
# Question 5: Feature Elimination 

# --- Question 5: Feature Elimination ---
features_all = list(df_train.columns)
target_subset = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
differences = {}

print(f"Baseline accuracy (unrounded): {original_acc:.5f}\n")

for f in target_subset:
    features_sub = [col for col in features_all if col != f]
    
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(df_train[features_sub].to_dict(orient='records'))
    X_val_sub = dv_sub.transform(df_val[features_sub].to_dict(orient='records'))
    
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    sub_acc = accuracy_score(y_val, model_sub.predict(X_val_sub))
    diff = original_acc - sub_acc
    differences[f] = diff
    
    print(f"Without '{f}': acc = {sub_acc:.5f} | diff = {diff:+.5f}")

min_diff_feat = min(differences, key=differences.get)
print(f"\nFeature with smallest difference: {min_diff_feat} ({differences[min_diff_feat]:+.5f})")

Baseline accuracy (unrounded): 0.64500

Without 'lead_source': acc = 0.64200 | diff = +0.00300
Without 'number_of_courses_viewed': acc = 0.64300 | diff = +0.00200
Without 'interaction_count': acc = 0.60100 | diff = +0.04400

Feature with smallest difference: number_of_courses_viewed (+0.00200)


In [23]:
# Question 6: Regularization Parameter C Tuning 
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {}

for c in c_values:
    model_c = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)
    
    y_pred_c = model_c.predict(X_val)
    acc = accuracy_score(y_val, y_pred_c)
    acc_rounded = round(acc, 3)
    c_scores[c] = acc_rounded
    
    print(f"C: {c:<8} -> Raw Acc: {acc:.5f} | round(3): {acc_rounded}")

best_acc = max(c_scores.values())
# If there is a tie, select the smallest C
best_c = min([c for c, acc in c_scores.items() if acc == best_acc])

print(f"\nBest C: {best_c} with accuracy {best_acc}")

C: 1e-06    -> Raw Acc: 0.59800 | round(3): 0.598
C: 1e-05    -> Raw Acc: 0.59800 | round(3): 0.598
C: 0.0001   -> Raw Acc: 0.61300 | round(3): 0.613
C: 0.001    -> Raw Acc: 0.64500 | round(3): 0.645

Best C: 0.001 with accuracy 0.645


# Machine Learning Zoomcamp 2026 — Week 3 Homework: Classification

---

## Summary of Answers

| Question | Topic / Task | Result / Answer | Metric / Value |
|---|---|---|---|
| **Q1** | Most frequent observation (mode) for `industry` | **`technology`** | Count = 1,173 |
| **Q2** | Two numerical features with the highest correlation | **`interaction_count and lead_score`** | Pearson $r \approx 0.9157$ |
| **Q3** | Categorical feature with biggest mutual information | **`lead_source`** | MI score = `0.03` |
| **Q4** | Baseline validation accuracy of Logistic Regression | **`0.65`** | Raw = `0.6450` |
| **Q5** | Feature elimination: smallest accuracy difference | **`'number_of_courses_viewed'`** | Difference = `+0.00200` |
| **Q6** | Value of parameter $C$ giving the best accuracy | **`0.001`** | Accuracy = `0.645` |

---

## Detailed Breakdown

### Data Preparation
* Categorical columns (`lead_source`, `industry`, `employment_status`, `location`) filled with `'NA'`.
* Numerical columns (`annual_income`, `number_of_courses_viewed`, `interaction_count`, `lead_score`) filled with `0.0`.
* Target: `converted`.

### Question 1: Mode of Industry
* Frequency counts: `technology` (1,173), `retail` (925), `healthcare` (840), `finance` (720), `education` (639), `manufacturing` (463), `NA` (240).
* **Selected Option:** **`technology`**.

### Question 2: Numerical Feature Correlation
* Tested candidate pairs:
  * `interaction_count` & `lead_score`: **`0.9157`**
  * `number_of_courses_viewed` & `lead_score`: `0.7572`
  * `number_of_courses_viewed` & `interaction_count`: `0.7216`
  * `annual_income` & `interaction_count`: `0.1228`
* **Selected Option:** **`interaction_count and lead_score`**.

### Question 3: Mutual Information
* Evaluated against `y_train` using training set only:
  * `lead_source`: **`0.03`**
  * `employment_status`: `0.02`
  * `industry`: `0.00`
  * `location`: `0.00`
* **Selected Option:** **`lead_source`**.

### Question 4: Baseline Logistic Regression
* Vectorized with `DictVectorizer(sparse=False)`.
* Model: `LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`.
* Validation Accuracy: `0.6450` $\rightarrow$ rounded to 2 decimals is **`0.65`**.
* **Selected Option:** **`0.65`**.

### Question 5: Feature Elimination
* Baseline accuracy: `0.64500`.
* Accuracy differences without each feature:
  * Without `lead_source`: `diff = +0.00300` (acc = `0.64200`)
  * Without `number_of_courses_viewed`: `diff = +0.00200` (acc = `0.64300`)
  * Without `interaction_count`: `diff = +0.04400` (acc = `0.60100`)
* **Selected Option:** **`'number_of_courses_viewed'`**.

### Question 6: Regularization Parameter $C$ Tuning
* Validation accuracy across candidate $C$ values:
  * $C = 0.000001 \rightarrow 0.598$
  * $C = 0.00001 \rightarrow 0.598$
  * $C = 0.0001 \rightarrow 0.613$
  * $C = 0.001 \rightarrow 0.645$
* **Selected Option:** **`0.001`**.